In [10]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [2]:
og_df = pd.read_csv('regional_stats.csv').iloc[:, 1:]
og_df

,country,state,lpopd,egood,ebad,mining,plantations,enone,yppp,lyppp,...,lseats,native,black,temp_avg,temp2,rainfall,rain2,alti,alti2,landlocked
0,Argentina,Buenos Aires,-2.974981,1,0,0.0,0.0,0,9321.192383,9.140046,...,-12.095780,2.508355,NaN,15.875000,252.01562,1.0926,1.193775,0.026,0.000676,0.0
1,Argentina,Catamarca,-1.790302,1,0,0.0,0.0,0,7304.713379,8.896275,...,-10.877821,2.522629,NaN,20.525000,421.27560,0.4581,0.209856,0.519,0.269361,1.0
2,Argentina,Chaco,-0.462096,1,0,0.0,0.0,0,5624.872070,8.634953,...,-11.645584,3.627478,NaN,21.008333,441.35007,1.5567,2.423315,0.047,0.002209,1.0
3,Argentina,Chubut,-4.711761,0,0,0.0,0.0,1,13967.312500,9.544475,...,-11.080855,9.685770,NaN,13.050000,170.30250,0.2283,0.052121,0.003,0.000009,0.0
4,Argentina,Ciudad de Buenos Aires (Capital Federal),-2.974981,1,0,0.0,0.0,0,30949.996094,10.340128,...,-11.693580,2.316357,NaN,17.725000,314.17563,1.2146,1.475253,0.010,0.000100,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
340,Venezuela,Portuguesa,0.367317,1,0,0.0,0.0,0,3496.789062,8.159600,...,-11.498904,0.200000,NaN,26.500000,702.25000,1.6050,2.576025,0.172,0.029584,1.0
341,Venezuela,Sucre,1.023031,0,0,0.0,0.0,1,4276.005859,8.360775,...,-11.417274,1.000000,NaN,26.850000,720.92255,0.7280,0.529984,0.060,0.003600,0.0
342,Venezuela,Trujillo,0.576342,1,0,0.0,0.0,0,4283.611328,8.362552,...,-11.468277,0.200000,NaN,25.100000,630.01000,1.1870,1.408969,0.884,0.781456,1.0
343,Venezuela,Táchira,0.431626,1,0,0.0,0.0,0,5822.441895,8.669475,...,-11.527893,0.200000,NaN,24.866667,618.35114,1.4970,2.241009,1.213,1.471369,1.0


In [24]:
""" Replicate Table 1. Regional PPP GDP per Capita across the Americas"""
yppp_stat_region = og_df.groupby("country").agg(
    Observations=("yppp", "count"),
    Mean=("yppp", "mean"),
    Maximum=("yppp", "max"), 
    Minimum=("yppp", "min")
)
yppp_stat_region["Log S.D."] = og_df.groupby("country")["lyppp"].std()
yppp_stat_region["Ratio y max / y min"] = yppp_stat_region["Maximum"] / yppp_stat_region["Minimum"]

# rearranging and rounding to match the og table
yppp_stat_region = yppp_stat_region[['Observations', 'Mean', 'Log S.D.', 'Minimum', 'Maximum', 'Ratio y max / y min']]
yppp_stat_region = yppp_stat_region.round({'Log S.D.': 3, 'Ratio y max / y min': 2})
int_cols = ['Observations', 'Mean', 'Minimum', 'Maximum']
yppp_stat_region[int_cols] = yppp_stat_region[int_cols].astype(int)
yppp_stat_region

,Observations,Mean,Log S.D.,Minimum,Maximum,Ratio y max / y min
country,,,,,,
Argentina,24,11705,0.553,4578,40450,8.84
Bolivia,9,2715,0.395,1244,4222,3.39
Brazil,27,5753,0.576,1792,17595,9.81
Canada,13,44267,0.358,26941,94900,3.52
Chile,13,8728,0.423,4154,19820,4.77
Colombia,30,5868,0.489,2367,22314,9.43
Ecuador,22,5057,0.834,1457,26573,18.23
El Salvador,12,3336,0.300,2191,5954,2.72
Guatemala,8,3562,0.439,2100,8400,4.00


In [26]:
yppp_stat_region.to_latex(
    "table1_regional_gdp.tex",
    index=True,
    caption="Regional PPP GDP per Capita across the Americas",
    label="tab:regional_gdp"
)

In [25]:
""" Replicate Table 2. Summary Statistics"""
t2_cols = [
    'lyppp', 'lpoverty', 'lhealth', 'lgini', 'lschoolspk', 'llit', 'lseats',
    'egood', 'ebad', 'mining', 'plantations', 'enone', 'lpopd',
    'temp_avg', 'temp2', 'rainfall', 'rain2', 'alti', 'alti2', 'landlocked'
]
new_var_map = {
    'lyppp': 'Log PPP GDP per capita',
    'lpoverty': 'Log poverty rate',
    'lhealth': 'Health Index',
    'lgini': 'Log Gini',
    'lschoolspk': 'Log schools per child',
    'llit': 'Log literacy rate',
    'lseats': 'Log seats in lower house per voter',
    'egood': 'Good activities dummy',
    'ebad': 'Bad activities dummy',
    'mining': 'Mining dummy',
    'plantations': 'Plantations dummy',
    'enone': 'No activities dummy',
    'lpopd': 'Log precolonial population density',
    'temp_avg': 'Average temperature',
    'temp2': 'Average temperature squared',
    'rainfall': 'Total rainfall',
    'rain2': 'Total rainfall squared',
    'alti': 'Altitude',
    'alti2': 'Altitude squared',
    'landlocked': 'Landlocked dummy'
}
new_stat_map = {
    'count': 'Observations',
    'mean': 'Mean',
    'std': 'S.D.',
    'min': 'Minimum',
    'max': 'Maximum'
}

# generate stats, rename columns and rows
summary_stats = og_df[t2_cols].agg(['count', 'mean', 'std', 'min', 'max'])
summary_stats = summary_stats.rename(columns=new_var_map, index=new_stat_map)

# transpose and round to match og table
summary_stats = summary_stats.T
summary_stats = summary_stats.round({'Mean' : 2, 'S.D.' : 2, 'Minimum' : 2, 'Maximum' : 2})
summary_stats['Observations'] = summary_stats['Observations'].astype(int)
summary_stats.index.name = 'Outcome Variables'
summary_stats

,Observations,Mean,S.D.,Minimum,Maximum
Outcome Variables,,,,,
Log PPP GDP per capita,345,8.83,0.97,7.13,11.67
Log poverty rate,331,2.93,0.92,0.21,4.40
Health Index,53,4.22,0.38,2.95,4.52
Log Gini,268,-0.74,0.16,-1.15,-0.46
Log schools per child,317,-5.31,0.64,-7.29,-3.69
Log literacy rate,270,-0.14,0.13,-0.76,-0.00
Log seats in lower house per voter,318,-11.42,1.19,-13.59,-8.14
Good activities dummy,345,0.61,0.49,0.00,1.00
Bad activities dummy,345,0.21,0.41,0.00,1.00


In [27]:
summary_stats.to_latex(
    "table2_summary_stats.tex",
    index=True,
    caption="Summary Statistics",
    label="tab:summary_stats"
)

In [32]:
""" Table 4 Panel A"""

# define independant variables for regression
colo_regr_df = og_df.copy()
X_cols = ['lpopd', 'temp_avg', 'temp2', 'rainfall', 'rain2', 'alti', 'alti2', 'landlocked']
X = sm.add_constant(colo_regr_df[X_cols])

# create colonial activity columns
colo_regr_df["bad"]  = ((colo_regr_df["mining"] == 1) |
                        (colo_regr_df["plantations"] == 1)).astype(int)
colo_regr_df["none"] = colo_regr_df["enone"]

pop_cut = colo_regr_df["lpopd"].median()
colo_regr_df["good"] = ((colo_regr_df["bad"] == 0) &
                        (colo_regr_df["none"] == 0) &
                        (colo_regr_df["lpopd"] <= pop_cut)).astype(int)
colo_regr_df["ugly"] = ((colo_regr_df["bad"] == 0) &
                        (colo_regr_df["none"] == 0) &
                        (colo_regr_df["lpopd"]  > pop_cut)).astype(int)

y_vars = {
    "bad": "Bad Activities",
    "mining": "Mining",
    "plantations": "Plantations",
    "good": "Good Activities",
    "ugly": "Ugly Activities",
    "none": "No Activities"
}

# run regression
panel_a_results = {}
for col, label in y_vars.items():
    y = colo_regr_df[col]
    panel_a_results[label] = sm.OLS(y, X).fit()
with open("panel_a_regression_output.txt", "w") as f:
    for label, result in panel_a_results.items():
        f.write(f"\n===== Regression for: {label} =====\n")
        f.write(result.summary().as_text())
        f.write("\n\n")

In [33]:
""" Table 4 Panel B"""

# define independant variables, add country dummies
base_X = ['lpopd','temp_avg','temp2','rainfall','rain2','alti','alti2','landlocked']
country_dummies = pd.get_dummies(colo_regr_df['country'], drop_first=True, dtype=float)
X = sm.add_constant(pd.concat([colo_regr_df[base_X], country_dummies], axis=1)).astype(float)

# run regression
panel_b_results = {}
for col, label in y_vars.items():
    y = colo_regr_df[col].astype(float)
    panel_b_results[label] = sm.OLS(y, X).fit()

with open("panel_b_regression_output.txt", "w") as f:
    for label, result in panel_b_results.items():
        f.write(f"\n===== Panel B Regression for: {label} =====\n")
        f.write(result.summary().as_text())
        f.write("\n\n")


In [36]:
panel_b_results["Mining"].summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 mining   R-squared:                       0.261
Model:                            OLS   Adj. R-squared:                  0.206
Method:                 Least Squares   F-statistic:                     4.718
Date:                Thu, 01 May 2025   Prob (F-statistic):           2.80e-11
Time:                        20:17:03   Log-Likelihood:                -51.615
No. Observations:                 345   AIC:                             153.2
Df Residuals:                     320   BIC:                             249.3
Df Model:                          24                                         
Covariance Type:            nonrobust                                         
===============================================================================
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const           0.0248      0.140      0.177      0.860      -0.251       0.300
lpopd          -0.0176      0.012     -1.441      0.150      -0.042       0.006
temp_avg        0.0091      0.012      0.743      0.458      -0.015       0.033
temp2          -0.0004      0.000     -1.092      0.275      -0.001       0.000
rainfall       -0.1191      0.044     -2.716      0.007      -0.205      -0.033
rain2           0.0277      0.008      3.567      0.000       0.012       0.043
alti           -0.0776      0.066     -1.175      0.241      -0.208       0.052
alti2           0.0646      0.019      3.329      0.001       0.026       0.103
landlocked     -0.0228      0.041     -0.561      0.575      -0.103       0.057
Bolivia        -0.0275      0.134     -0.206      0.837      -0.290       0.235
Brazil          0.2691      0.101      2.676      0.008       0.071       0.467
Canada          0.0363      0.143      0.254      0.800      -0.245       0.318
Chile           0.2343      0.108      2.159      0.032       0.021       0.448
Colombia        0.2108      0.106      1.986      0.048       0.002       0.420
Ecuador         0.2150      0.106      2.029      0.043       0.007       0.424
El Salvador     0.1914      0.132      1.453      0.147      -0.068       0.451
Guatemala       0.1304      0.144      0.906      0.366      -0.153       0.414
Honduras        0.5297      0.111      4.756      0.000       0.311       0.749
Mexico          0.2494      0.102      2.441      0.015       0.048       0.450
Panama          0.0661      0.141      0.468      0.640      -0.212       0.344
Paraguay        0.1621      0.099      1.629      0.104      -0.034       0.358
Peru            0.1171      0.107      1.095      0.274      -0.093       0.328
US             -0.0009      0.077     -0.012      0.991      -0.152       0.150
Uruguay        -0.0241      0.095     -0.253      0.801      -0.212       0.164
Venezuela       0.2297      0.111      2.061      0.040       0.010       0.449
==============================================================================
Omnibus:                      113.757   Durbin-Watson:                   1.993
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              270.798
Skew:                           1.642   Prob(JB):                     1.57e-59
Kurtosis:                       5.838   Cond. No.                     9.80e+03
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 9.8e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [44]:
""" generate tables to match the paper"""

def star_sig(p):
    """return significance stars."""
    if p < 0.01:  return '***'
    if p < 0.05:  return '**'
    if p < 0.10:  return '*'
    return ''

def generate_panel(results):
    # columns and rows 
    col_order = ["Bad Activities", "Mining", "Plantations", "Good Activities", "Ugly Activities", "No Activities"]
    var_order = [
        ('lpopd',     "Log precolonial population density"),    # rename
        ('temp_avg',  "Average temperature"),
        ('temp2',     "Average temperature squared"),
        ('rainfall',  "Total rainfall"),
        ('rain2',     "Total rainfall squared"),
        ('alti',      "Altitude"),
        ('alti2',     "Altitude squared"),
        ('landlocked',"Landlocked dummy")
    ]
    
    output_df = pd.DataFrame(index=[ind for _, ind in var_order],
                   columns=col_order, dtype=str)

    for out in col_order:
        res = results[out]                 # statsmodels result
        for var, pretty in var_order:
            coef  = res.params[var]
            se    = res.bse[var]
            cell  = f"{coef:.3f}{star_sig(res.pvalues[var])}\n({se:.3f})"
            output_df.loc[pretty, out] = cell

    # add R_2 row
    r2_row = {out: f"{results[out].rsquared:.3f}" for out in col_order}
    output_df.loc["$R^2$"] = r2_row

    return output_df

In [47]:
generate_panel(panel_a_results).to_latex("table4_panelA.tex", escape=False)
generate_panel(panel_b_results).to_latex("table4_panelB.tex", escape=False)